# Laboratorio 9. Series de tiempo y forecasting

## Notebook 2. Investigación de herramientas para forecasting en PyTorch

Este notebook cubre la sección 3 del enunciado. Para cada clase o función se explica para qué sirve, cuáles son sus parámetros más importantes y cuál es la forma de sus entradas y salidas. Las formas se comprueban con código corto que corre al final de cada apartado. Después vienen los conceptos que pide el enunciado (modelo local contra global, teacher forcing, RevIN, pinball loss y MASE contra WAPE). Los links de cada apartado llevan a la documentación oficial o al paper original.

In [1]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("torch", torch.__version__)

torch 2.11.0+cu128


### nn.LSTM y nn.GRU

Documentación oficial en https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html y https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html. Los papers originales son Hochreiter y Schmidhuber (1997) para la LSTM y Cho et al. (2014) para la GRU, en https://arxiv.org/abs/1406.1078.

Las dos son capas recurrentes. Leen la secuencia un paso a la vez y guardan un estado oculto que resume lo que han visto. La LSTM tiene además una celda de memoria `c` que se controla con tres compuertas (olvido, entrada y salida). La GRU junta todo en un solo estado y usa dos compuertas (reinicio y actualización), entonces tiene menos parámetros y suele entrenar un poco más rápido.

| Parámetro | Para qué sirve |
| --- | --- |
| `input_size` | Cantidad de features en cada paso de la secuencia. En este laboratorio son 8, la serie normalizada y las 7 covariables de calendario. |
| `hidden_size` | Tamaño del estado oculto `h` (y de la celda `c` en la LSTM). Es el que más pesa en el número de parámetros. |
| `num_layers` | Cantidad de capas apiladas. La salida de una capa es la entrada de la siguiente. |
| `batch_first` | Si es `True` los tensores son `(batch, secuencia, features)`. Si es `False` (el valor por defecto) son `(secuencia, batch, features)`. |
| `dropout` | Probabilidad de apagar unidades en la salida de cada capa, menos la última. Solo hace algo si `num_layers` es mayor que 1. |
| `bidirectional` | Si es `True` lee la secuencia en los dos sentidos y duplica el tamaño de la salida. No sirve para pronosticar hacia adelante con la salida completa, porque el sentido inverso ve el futuro de la ventana. |

Las salidas de la LSTM son `output, (h_n, c_n)` y las de la GRU son `output, h_n`.

- `output` tiene forma `(batch, L, D * hidden_size)` con `batch_first=True`. Trae el estado oculto de la última capa en todos los pasos. `D` es 2 si la capa es bidireccional y 1 si no.
- `h_n` tiene forma `(D * num_layers, batch, hidden_size)`. Es el estado oculto del último paso de cada capa.
- `c_n` tiene la misma forma que `h_n` y es el estado de la celda en el último paso. Solo existe en la LSTM.

Para pronosticar se usa el último paso, o sea `output[:, -1]`, que es igual a `h_n[-1]` cuando la capa no es bidireccional. En la estrategia autorregresiva ese estado se vuelve a pasar a la red paso a paso, y en la multi-salida una capa densa saca las 24 horas de una vez.

In [2]:
B, L, k, H = 4, 168, 8, 64
x = torch.randn(B, L, k)

lstm = nn.LSTM(input_size=k, hidden_size=H, num_layers=2, batch_first=True, dropout=0.1)
out, (h_n, c_n) = lstm(x)
print("LSTM  output", tuple(out.shape), " h_n", tuple(h_n.shape), " c_n", tuple(c_n.shape))

gru = nn.GRU(input_size=k, hidden_size=H, num_layers=2, batch_first=True, dropout=0.1)
out_g, h_g = gru(x)
print("GRU   output", tuple(out_g.shape), " h_n", tuple(h_g.shape))

print("output[:, -1] igual a h_n[-1] en la LSTM:", torch.allclose(out[:, -1], h_n[-1]))

n_lstm = sum(p.numel() for p in lstm.parameters())
n_gru = sum(p.numel() for p in gru.parameters())
print(f"parámetros con la misma configuración  LSTM {n_lstm:,}  GRU {n_gru:,}  (razón {n_gru / n_lstm:.2f}, o sea 3 de 4)")

LSTM  output (4, 168, 64)  h_n (2, 4, 64)  c_n (2, 4, 64)


GRU   output (4, 168, 64)  h_n (2, 4, 64)


output[:, -1] igual a h_n[-1] en la LSTM: True
parámetros con la misma configuración  LSTM 52,224  GRU 39,168  (razón 0.75, o sea 3 de 4)


### nn.Conv1d con dilation y padding causal, y el campo receptivo de una TCN

Documentación oficial en https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv1d.html. La TCN viene de Bai, Kolter y Koltun (2018) en https://arxiv.org/abs/1803.01271.

`nn.Conv1d` recibe un tensor `(batch, canales, tiempo)` y desliza filtros a lo largo del tiempo. Los parámetros más importantes son `in_channels`, `out_channels`, `kernel_size` (cuántos pasos mira cada filtro), `stride` (de a cuántos pasos avanza), `padding` (ceros que agrega en los bordes) y `dilation` (separación entre los pasos que mira el filtro). Con `dilation=d` el filtro mira los pasos `t`, `t-d`, `t-2d` y así. La longitud de salida es

`L_out = floor((L + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1)`

Una convolución normal con `padding` simétrico deja que la salida de `t` use pasos futuros, y eso en forecasting sería una fuga. Para que sea causal se agregan ceros solo a la izquierda, `(kernel_size - 1) * dilation`, con `F.pad`, y se deja `padding=0` en la convolución. Así la salida de `t` depende solo de los pasos `t` y anteriores, y la longitud no cambia.

El campo receptivo es cuántos pasos de la entrada alcanza a ver una salida. Una capa con kernel `k` y dilatación `d` agrega `(k - 1) * d` pasos. Si se apilan `n` capas con dilatación que se duplica (1, 2, 4, ...) y una convolución por capa, el campo receptivo es

`RF = 1 + (k - 1) * (2^n - 1)`

La TCN original usa dos convoluciones por bloque residual, entonces cada bloque aporta el doble y queda `RF = 1 + 2 * (k - 1) * (2^n - 1)`. La dilatación exponencial hace que el campo receptivo crezca de forma exponencial con la cantidad de capas, mientras que los parámetros crecen de forma lineal.

In [3]:
def convolucion_causal(x, conv):
    # ceros solo a la izquierda, así la salida en t no mira pasos futuros
    relleno = (conv.kernel_size[0] - 1) * conv.dilation[0]
    return conv(F.pad(x, (relleno, 0)))


def campo_receptivo(kernel, n_capas, convs_por_bloque=1):
    return 1 + convs_por_bloque * (kernel - 1) * (2 ** n_capas - 1)


x = torch.randn(2, 1, 200)
conv = nn.Conv1d(1, 4, kernel_size=3, dilation=4, padding=0)
y = convolucion_causal(x, conv)
print("longitud de entrada", x.shape[-1], " longitud de salida", y.shape[-1])

# la salida en t no cambia si se altera el futuro de la entrada
x2 = x.clone()
x2[..., 120:] += 10.0
y2 = convolucion_causal(x2, conv)
print("salidas antes del paso 120 iguales:", torch.allclose(y[..., :120], y2[..., :120]),
      " después cambian:", not torch.allclose(y[..., 120:], y2[..., 120:]))

# campo receptivo medido contra la fórmula, con dilataciones 1, 2, 4, 8
capas = [nn.Conv1d(1, 1, 3, dilation=2 ** i) for i in range(4)]
z = torch.zeros(1, 1, 60, requires_grad=True)
h = z
for c in capas:
    h = convolucion_causal(h, c)
h[0, 0, -1].backward()
medido = (z.grad[0, 0] != 0).sum().item()
print("campo receptivo medido con 4 capas y kernel 3:", medido, " fórmula:", campo_receptivo(3, 4))

for n in range(4, 9):
    print(f"{n} capas, kernel 3  una conv por capa RF = {campo_receptivo(3, n):4d}   dos convs por bloque RF = {campo_receptivo(3, n, 2):4d}")
print("para ver una semana (168 horas) con kernel 3 bastan 7 capas con una conv, o 6 bloques con dos convs")

longitud de entrada 200  longitud de salida 200
salidas antes del paso 120 iguales: True  después cambian: True


campo receptivo medido con 4 capas y kernel 3: 31  fórmula: 31
4 capas, kernel 3  una conv por capa RF =   31   dos convs por bloque RF =   61
5 capas, kernel 3  una conv por capa RF =   63   dos convs por bloque RF =  125
6 capas, kernel 3  una conv por capa RF =  127   dos convs por bloque RF =  253
7 capas, kernel 3  una conv por capa RF =  255   dos convs por bloque RF =  509
8 capas, kernel 3  una conv por capa RF =  511   dos convs por bloque RF = 1021
para ver una semana (168 horas) con kernel 3 bastan 7 capas con una conv, o 6 bloques con dos convs


### nn.TransformerEncoderLayer y la codificación posicional

Documentación oficial en https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html. La arquitectura viene de Vaswani et al. (2017) en https://arxiv.org/abs/1706.03762.

Una capa del encoder tiene dos partes. Primero una autoatención de varias cabezas, donde cada paso de la secuencia mira a todos los demás y decide cuánto pesa cada uno. Después una red feed-forward de dos capas lineales que se aplica a cada paso por separado. Las dos partes llevan conexión residual y `LayerNorm`.

| Parámetro | Para qué sirve |
| --- | --- |
| `d_model` | Tamaño del vector que representa cada paso. Es la dimensión de entrada y de salida de la capa. |
| `nhead` | Número de cabezas de atención. `d_model` tiene que ser divisible entre `nhead`, y cada cabeza trabaja con `d_model / nhead` dimensiones. |
| `dim_feedforward` | Tamaño de la capa oculta de la red feed-forward (2048 por defecto). |
| `dropout` | Dropout dentro de la capa (0.1 por defecto). |
| `batch_first` | Con `True` el tensor es `(batch, secuencia, d_model)`. Por defecto es `False` y el tensor es `(secuencia, batch, d_model)`. |
| `norm_first` | Si es `True` aplica `LayerNorm` antes de cada subcapa (pre-norm), que suele entrenar más estable. |

La autoatención no sabe en qué orden vienen los pasos, porque si se mezclan las posiciones las salidas se mezclan igual. Por eso la capa no trae posición y hay que sumarla a la entrada. Primero una capa lineal lleva las features de cada paso a `d_model`, y luego se suma (no se concatena) un vector de posición del mismo tamaño. Puede ser sinusoidal y fija,

`PE(pos, 2i) = sin(pos / 10000^(2i / d_model))` y `PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))`

o aprendida con un `nn.Embedding(L, d_model)`. En series de tiempo también se usa agrupar los pasos en parches antes de la atención, que es lo que hace PatchTST (https://arxiv.org/abs/2211.14730), y en iTransformer (https://arxiv.org/abs/2310.06625) se aplica la atención entre las series y no entre los pasos.

In [4]:
class CodificacionPosicional(nn.Module):
    def __init__(self, d_model, largo_max=512):
        super().__init__()
        pos = torch.arange(largo_max).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(largo_max, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.shape[1]]


d_model, nhead = 64, 4
entrada = nn.Linear(8, d_model)
posicion = CodificacionPosicional(d_model)
capa = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128,
                                  dropout=0.1, batch_first=True, norm_first=True)
encoder = nn.TransformerEncoder(capa, num_layers=2)

x = torch.randn(4, 168, 8)
z = posicion(entrada(x))
y = encoder(z)
print("entrada", tuple(x.shape), "-> con posición", tuple(z.shape), "-> salida del encoder", tuple(y.shape))

# sin posición la capa trata igual a una secuencia y a la misma secuencia barajada
encoder.eval()
with torch.no_grad():
    z0 = entrada(x)
    perm = torch.randperm(168)
    sin_pos = encoder(z0)[:, perm]
    barajada = encoder(z0[:, perm])
    con_pos = encoder(posicion(z0))[:, perm]
    con_pos_b = encoder(posicion(z0[:, perm]))
print("sin posición, barajar la entrada solo baraja la salida:", torch.allclose(sin_pos, barajada, atol=1e-5))
print("con posición, la salida sí cambia al barajar:", not torch.allclose(con_pos, con_pos_b, atol=1e-5))

entrada (4, 168, 8) -> con posición (4, 168, 64) -> salida del encoder (4, 168, 64)


sin posición, barajar la entrada solo baraja la salida: True


con posición, la salida sí cambia al barajar: True


### NeuralForecast y StatsForecast de Nixtla

Documentación en https://nixtlaverse.nixtla.io/statsforecast/ y https://nixtlaverse.nixtla.io/neuralforecast/. El código está en https://github.com/Nixtla/statsforecast y https://github.com/Nixtla/neuralforecast. Para el modelo de árboles se usa MLForecast, documentado en https://nixtlaverse.nixtla.io/mlforecast/.

StatsForecast tiene modelos estadísticos rápidos (`Naive`, `SeasonalNaive`, `MSTL`, `AutoETS`) y NeuralForecast tiene modelos de redes (`NHITS`, `TCN`, `PatchTST`, `iTransformer`, `DeepAR`). Las dos reciben los datos en formato largo, con una fila por serie y por paso de tiempo y tres columnas.

| Columna | Contenido |
| --- | --- |
| `unique_id` | Identificador de la serie, aquí el código del cliente (`MT_001`). |
| `ds` | Fecha y hora del paso, aquí el inicio de cada hora. |
| `y` | Valor a pronosticar, aquí los kWh de esa hora. |

Cualquier otra columna se toma como variable exógena. Las covariables que se conocen a futuro, como el calendario, entran a NeuralForecast con `futr_exog_list` y hay que pasar sus valores futuros en `futr_df` al predecir. La frecuencia se da con `freq="h"`.

El método `cross_validation` hace un backtest con origen móvil en una sola llamada. Los parámetros importantes son estos.

| Parámetro | Qué hace |
| --- | --- |
| `h` | Horizonte, cuántos pasos se pronostican en cada ventana. Aquí 24. |
| `n_windows` | Cantidad de ventanas de evaluación, contadas hacia atrás desde el final de los datos. Aquí 92, una por día de prueba. |
| `step_size` | Cuántos pasos se mueve el origen entre una ventana y la siguiente. Aquí 24, o sea un día. |
| `refit` | Si es `True`, StatsForecast ajusta otra vez los modelos en cada ventana. Si es `False` ajusta una vez, con los datos hasta el primer corte, y usa ese ajuste para todas las ventanas. Un entero reajusta cada tantas ventanas. En NeuralForecast el valor por defecto es `False`, porque reentrenar una red en cada ventana sale muy caro. |
| `val_size`, `test_size` | Solo en NeuralForecast. `val_size` reserva las últimas observaciones de entrenamiento para validación y early stopping, y `test_size` es la cantidad de pasos finales que se evalúan. |
| `input_size` | En StatsForecast limita cuántos pasos de historia usa cada ajuste. En los modelos de NeuralForecast es la ventana de entrada L. |

La salida trae `unique_id`, `ds`, `cutoff` (la última hora que vio el modelo, aquí las 23:00 del día anterior), `y` y una columna por modelo. El origen del pronóstico es una hora después del `cutoff`, y eso se revisa en el código antes de pasar los resultados por `backtest`.

En este laboratorio los modelos de NeuralForecast se entrenan con `fit` (con `val_size = 2208`, o sea los 92 días de validación) y se predice origen por origen con `predict`, para que pasen por el mismo `backtest` que todos los demás.

In [5]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive

from src import config as C
from src.data import cargar_panel

panel = cargar_panel()
pequeno = panel[panel.unique_id.isin(["MT_013", "MT_046"])]
print(pequeno.head(3).to_string(index=False))

cv = StatsForecast(models=[SeasonalNaive(24)], freq="h").cross_validation(
    df=pequeno, h=24, step_size=24, n_windows=3, refit=False)
print()
print(cv.head(3).to_string(index=False))
print("cortes", cv.cutoff.unique().tolist())
print("filas por serie y ventana", cv.groupby(["unique_id", "cutoff"]).size().unique().tolist(), "(h = 24)")

unique_id                  ds         y
   MT_013 2012-01-01 00:00:00 47.678276
   MT_013 2012-01-01 01:00:00 46.019900
   MT_013 2012-01-01 02:00:00 43.532339

unique_id                  ds              cutoff         y  SeasonalNaive
   MT_013 2014-12-29 00:00:00 2014-12-28 23:00:00 38.971808      38.764511
   MT_013 2014-12-29 01:00:00 2014-12-28 23:00:00 33.167496      34.825871
   MT_013 2014-12-29 02:00:00 2014-12-28 23:00:00 33.582090      34.411277
cortes [Timestamp('2014-12-28 23:00:00'), Timestamp('2014-12-29 23:00:00'), Timestamp('2014-12-30 23:00:00')]
filas por serie y ventana [24] (h = 24)


### BaseChronosPipeline.predict_quantiles de Chronos

Código en https://github.com/amazon-science/chronos-forecasting, paper de Chronos en https://arxiv.org/abs/2403.07815 y modelos en https://huggingface.co/amazon/chronos-bolt-base.

Chronos es una familia de modelos fundacionales para series de tiempo. Se entrenaron con una gran colección de series de muchos dominios, y por eso se pueden usar sin entrenamiento (zero-shot) sobre series nuevas. Chronos-Bolt es la versión rápida. En vez de generar muestras una por una, un encoder-decoder entrega los cuantiles directamente en una sola pasada.

El pipeline se carga con `BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base", device_map="cuda")`. El método `predict_quantiles` recibe estos argumentos.

| Argumento | Qué es |
| --- | --- |
| `inputs` | El contexto. Puede ser un tensor 1D (una serie), un tensor 2D `(batch, T)` o una lista de tensores 1D de distinto largo. Si la serie es más larga que el contexto del modelo, se usan solo los últimos pasos. |
| `prediction_length` | Cuántos pasos se pronostican. Aquí 24. |
| `quantile_levels` | Lista de cuantiles que se piden. Aquí `[0.1, 0.5, 0.9]`. Chronos-Bolt se entrenó con los nueve cuantiles de 0.1 a 0.9, y si se piden otros los interpola. |

Devuelve una tupla `(quantiles, mean)`. `quantiles` tiene forma `(batch, prediction_length, len(quantile_levels))` y `mean` tiene forma `(batch, prediction_length)`. En Chronos-Bolt la media que devuelve es la mediana (el cuantil 0.5). Con eso salen el pronóstico puntual (P50) y el intervalo de 80% (P10 a P90) de la sección de pronóstico probabilístico.

Chronos normaliza cada serie por dentro, entonces se le pasan los kWh originales y no los valores con z-score.

### Modelo local contra modelo global

Un modelo local entrena uno por serie, y en este laboratorio serían 100 modelos. Cada uno solo ve la historia de su cliente. Se adapta bien a la escala y a los hábitos de ese cliente, pero tiene pocos datos para aprender y hay que entrenar y mantener muchos modelos. Un modelo global entrena uno solo con todas las series. Comparte lo que se repite entre clientes (el ciclo de las 24 horas, la semana y los feriados) y tiene muchos más ejemplos, pero tiene que lidiar con series de escalas muy distintas. Por eso aquí se normaliza cada cliente con su z-score de entrenamiento. Los modelos locales ganan cuando las series se parecen poco entre sí, y los globales ganan cuando hay muchas series con estructura compartida. La sección 5.3 mide cuál conviene en estos datos.

### Teacher forcing

Es una forma de entrenar modelos autorregresivos y seq2seq. En cada paso del decoder se le da como entrada el valor real del paso anterior y no la predicción del propio modelo. Entrena más rápido y más estable, porque un error al inicio no arrastra a todos los pasos siguientes. El problema es que al pronosticar no hay valores reales, y el modelo tiene que usar sus propias predicciones, que nunca vio en entrenamiento. A esa diferencia se le llama exposure bias, y hace que los errores se acumulen con el horizonte. Una solución intermedia es dar el valor real con una probabilidad `p` y la predicción propia con probabilidad `1 - p` (scheduled sampling). El seq2seq de este laboratorio prueba `p = 0`, `0.5` y `1`. El trabajo que formalizó la idea es Williams y Zipser (1989), y el seq2seq viene de Sutskever et al. (2014) en https://arxiv.org/abs/1409.3215.

### RevIN

RevIN (Reversible Instance Normalization) es de Kim et al. (2022), en https://openreview.net/forum?id=cGDAkQo1C0p. A cada ventana de entrada se le resta su media y se divide entre su desviación estándar, y después se aplican dos parámetros aprendibles por canal (una escala y un desplazamiento). La red trabaja siempre con ventanas de media 0 y desviación 1. Al final se deshace la transformación sobre la salida, con la misma media y desviación de esa ventana, y de ahí viene lo de reversible. Ayuda cuando cambia el nivel de la serie entre el entrenamiento y la prueba (distribution shift), por ejemplo un cliente que consume más en invierno que en verano, porque el modelo ya no tiene que aprender el nivel y solo aprende la forma. Es parecido al z-score por serie, pero se calcula con cada ventana y no con todo el entrenamiento.

### Pinball loss y cuantiles

Para un cuantil `tau` entre 0 y 1, con valor real `y` y predicción del cuantil `q`, la pinball loss es

`L = max(tau * (y - q), (tau - 1) * (y - q))`

Si la predicción queda por debajo del real, el error pesa `tau`, y si queda por encima pesa `1 - tau`. Entonces con `tau = 0.9` es más caro quedarse corto que pasarse, y la predicción que minimiza la pérdida es el cuantil 0.9 de la distribución. Con `tau = 0.5` es la mitad del error absoluto y su mínimo es la mediana. Promediar la pinball loss de varios cuantiles (aquí 0.1, 0.5 y 0.9) mide qué tan bien está toda la distribución del pronóstico, que es más que el punto central. La cobertura P10 a P90 es la fracción de valores reales que cae entre el cuantil 0.1 y el 0.9. Si los cuantiles están bien calibrados debería ser cerca de 80%. Más detalle en https://otexts.com/fpp3/distaccuracy.html.

### MASE contra WAPE

Las dos métricas no dependen de la escala, pero se calculan distinto.

`WAPE = suma(|error|) / suma(|real|)`, sobre todo el panel.

`MASE = MAE del modelo / MAE del seasonal naive en entrenamiento`, y se calcula serie por serie antes de promediar.

El WAPE divide entre el total real, entonces los clientes grandes pesan más y los pequeños casi no cuentan. Además se vuelve inestable si el total real está cerca de cero. El MASE (Hyndman y Koehler, 2006, en https://doi.org/10.1016/j.ijforecast.2006.03.001) compara cada cliente contra su propio naive estacional, entonces todos pesan parecido sin importar su tamaño. Un MASE menor que 1 significa que el modelo es mejor que repetir el día anterior (el seasonal naive de 24 horas medido en entrenamiento), y mayor que 1 significa peor. En este laboratorio el MASE es la métrica con la que se ordena la tabla, y el WAPE se reporta como segunda referencia porque es la más fácil de interpretar en kWh.

In [6]:
def pinball(y, q, tau):
    e = y - q
    return np.maximum(tau * e, (tau - 1) * e).mean()


rng = np.random.default_rng(0)
y = rng.gamma(shape=4.0, scale=10.0, size=100_000)   # una distribución con cola a la derecha
candidatos = np.linspace(y.min(), np.quantile(y, 0.999), 400)
for tau in (0.1, 0.5, 0.9):
    mejor = candidatos[np.argmin([pinball(y, c, tau) for c in candidatos])]
    print(f"tau {tau}  el mínimo de la pinball cae en {mejor:6.2f}   el cuantil real es {np.quantile(y, tau):6.2f}")

# con tau 0.5 la pinball es la mitad del MAE
print("pinball(0.5) =", round(pinball(y, np.median(y), 0.5), 4), "  MAE / 2 =", round(np.abs(y - np.median(y)).mean() / 2, 4))

tau 0.1  el mínimo de la pinball cae en  17.50   el cuantil real es  17.44


tau 0.5  el mínimo de la pinball cae en  36.83   el cuantil real es  36.82
tau 0.9  el mínimo de la pinball cae en  66.64   el cuantil real es  66.75
pinball(0.5) = 7.6987   MAE / 2 = 7.6987


### Resumen de las herramientas y dónde se usan

| Herramienta | Dónde se usa en el laboratorio |
| --- | --- |
| `nn.LSTM` | LSTM multi-salida, LSTM autorregresiva y encoder y decoder del seq2seq (modelos propios). |
| `nn.Conv1d` causal con dilatación | TCN de NeuralForecast. |
| `nn.TransformerEncoderLayer` y codificación posicional | Idea base de PatchTST e iTransformer de NeuralForecast. |
| StatsForecast | Naive, seasonal naive de 24 y 168 horas y MSTL. |
| MLForecast con LightGBM | Modelo global de árboles con rezagos 1 a 24 y 168 más calendario. |
| NeuralForecast | N-HiTS, TCN, PatchTST, iTransformer y DeepAR. |
| Chronos-Bolt | Pronóstico probabilístico zero-shot, solo inferencia. |

## Referencias

- Bai, S., Kolter, J. Z. y Koltun, V. (2018). An empirical evaluation of generic convolutional and recurrent networks for sequence modeling. https://arxiv.org/abs/1803.01271
- Cho, K. et al. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine translation. https://arxiv.org/abs/1406.1078
- Hyndman, R. J. y Koehler, A. B. (2006). Another look at measures of forecast accuracy. International Journal of Forecasting, 22(4). https://doi.org/10.1016/j.ijforecast.2006.03.001
- Hyndman, R. J. y Athanasopoulos, G. (2021). Forecasting, principles and practice, 3a edición. https://otexts.com/fpp3/
- Kim, T. et al. (2022). Reversible instance normalization for accurate time-series forecasting against distribution shift. ICLR. https://openreview.net/forum?id=cGDAkQo1C0p
- Liu, Y. et al. (2024). iTransformer, inverted transformers are effective for time series forecasting. https://arxiv.org/abs/2310.06625
- Nie, Y. et al. (2023). A time series is worth 64 words, long-term forecasting with transformers (PatchTST). https://arxiv.org/abs/2211.14730
- Ansari, A. F. et al. (2024). Chronos, learning the language of time series. https://arxiv.org/abs/2403.07815
- Sutskever, I., Vinyals, O. y Le, Q. V. (2014). Sequence to sequence learning with neural networks. https://arxiv.org/abs/1409.3215
- Vaswani, A. et al. (2017). Attention is all you need. https://arxiv.org/abs/1706.03762
- Documentación de PyTorch en https://docs.pytorch.org/docs/stable/
- Documentación de Nixtla en https://nixtlaverse.nixtla.io/
- Repositorio de Chronos en https://github.com/amazon-science/chronos-forecasting